#1. Install deps

In [1]:
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/NLP'):
        !git clone -b lab-08 https://github.com/AndrianaNahirna/NLP.git

    %cd /content/NLP

    !pip install -r requirements.txt -q
    !git fetch origin
    !git reset --hard origin/lab-08

    sys.path.append('/content/NLP')

from sentiment.src.topic_modeling import run_lsa, run_lda
from sentiment.src.topic_utils import get_top_words, get_top_documents
print("Середовище налаштовано.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Cloning into 'NLP'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 283 (delta 28), reused 41 (delta 19), pack-reused 194 (from 1)
Receiving objects: 100% (283/283), 2.14 MiB | 3.82 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/NLP
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.1/990.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 27.1 MB/s eta 0:00:00
HEAD is now at bf5246f Update topic utils
Середовище налаштовано.


#2. Data access

In [3]:
import pandas as pd

if 'google.colab' in sys.modules:
    FOLDER_ID = '1Z4ko8PYcLJOnnU98T6MTXLVYHnpMkHVK'

    # Завантаження датасету
    os.makedirs('/content/NLP/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/NLP/data/

    import glob
    csv_files = glob.glob('/content/NLP/data/**/processed_v3_lemma.csv', recursive=True)

    if csv_files:
        data_path = csv_files[0]
        df = pd.read_csv(data_path)
        print(f"Датасет завантажено. Кількість рядків: {len(df)}")
    else:
        print("Файл processed_v3_lemma.csv не знайдено.")
else:
    # Локальний шлях
    df = pd.read_csv('../data/processed_v3_lemma.csv')

Retrieving folder contents
Processing file 12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI processed_v2
Processing file 17odn4ukdHLvZKqqUuaTNPuHX66Aal-zk processed_v2.csv
Processing file 1gMJmeUiP3HXGR4P3F3Gq-eWhKkPxAbnw processed_v3_lemma.csv
Processing file 1tVj7OaRkYqaoVtmDGgDxUQ8nkDUvy7W7 raw.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI
From (redirected): https://docs.google.com/spreadsheets/d/12MwPw-0rT5kZoMFme6erhb4NeBsQxsoECoQzPuAqWcI/export?format=xlsx
To: /content/NLP/data/NLP_datasets/processed_v2
1.60MB [00:00, 16.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=17odn4ukdHLvZKqqUuaTNPuHX66Aal-zk
To: /content/NLP/data/NLP_datasets/processed_v2.csv
100% 5.67M/5.67M [00:00<00:00, 48.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1gMJmeUiP3HXGR4P3F3Gq-eWhKkPxAbnw
To: /content/NLP/data/NLP_da

#3. Corpus filtering / preprocessing checks

In [18]:
# Кількість до фільтрації
len_before = len(df)

# Прибираємо порожні значення у колонці з лемами
df_filtered = df.dropna(subset=['lemma_text']).copy()

# Фільтруємо занадто короткі тексти (менше 20 символів)
df_filtered = df_filtered[df_filtered['lemma_text'].str.len() > 20]

# Фінальна кількість після фільтрації
len_after = len(df_filtered)

print(f"Кількість документів до фільтрації: {len_before}")
print(f"Кількість документів після фільтрації: {len_after}")
print(f"Видалено документів: {len_before - len_after}")

# Створюємо чистий список текстів для моделей
corpus = df_filtered['lemma_text'].astype(str).tolist()

Кількість документів до фільтрації: 3034
Кількість документів після фільтрації: 3034
Видалено документів: 0


#4. Vectorizer setup

In [19]:
!pip install -U spacy
!python -m spacy download uk_core_news_sm

import spacy
# Створюємо список стоп-слів
nlp = spacy.load("uk_core_news_sm")
stop_words = list(nlp.Defaults.stop_words)

# Визначаємо параметри фільтрації токенів
params = {
    'min_df': 5,      # ігноруємо слова, що зустрічаються менше ніж у 5 документах
    'max_df': 0.8,    # ігноруємо слова, що є у понад 80% документів
    'ngram_range_lsa': (1, 2), # для LSA беремо уніграми та біграми
    'ngram_range_lda': (1, 1)  # для LDA лише уніграми
}

print(f"Підготовлено {len(stop_words)} стоп-слів.")
print(f"Налаштовано параметри фільтрації: min_df={params['min_df']}, max_df={params['max_df']}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 74.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('uk_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Підготовлено 467 стоп-слів.
Налаштовано параметри фільтрації: min_df=5, max_df=0.8


#5. LSA experiments

In [21]:
from sentiment.src.topic_modeling import run_lsa

# Експеримент 1: LSA з k=5 темами
print("Запуск LSA (k=5)...")
lsa_model_5, lsa_vec_5, lsa_matrix_5 = run_lsa(
    corpus,
    stop_words,
    n_components=5
)

# Експеримент 2: LSA з k=8 темами
print("Запуск LSA (k=8)...")
lsa_model_8, lsa_vec_8, lsa_matrix_8 = run_lsa(
    corpus,
    stop_words,
    n_components=8
)

# Експеримент 2: LSA з k=10 темами
print("Запуск LSA (k=10)...")
lsa_model_10, lsa_vec_10, lsa_matrix_10 = run_lsa(
    corpus,
    stop_words,
    n_components=10
)

print("LSA експерименти завершено.")

Запуск LSA (k=5)...


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['дев', 'ятий', 'ятнадцятий', 'ятнадцять', 'ять'] not in stop_words.
  warnings.warn(


Запуск LSA (k=8)...
Запуск LSA (k=10)...
LSA експерименти завершено.


#6. LDA experiments

In [22]:
from sentiment.src.topic_modeling import run_lda

# Експеримент 1: LDA з k=5 темами
print("Запуск LDA (k=5)...")
lda_model_5, lda_vec_5, lda_matrix_5 = run_lda(
    corpus,
    stop_words,
    n_components=5
)

# Експеримент 2: LDA з k=8 темами
print("Запуск LDA (k=8)...")
lda_model_8, lda_vec_8, lda_matrix_8 = run_lda(
    corpus,
    stop_words,
    n_components=8
)

# Експеримент 3: LDA з k=10 темами
print("Запуск LDA (k=10)...")
lda_model_10, lda_vec_10, lda_matrix_10 = run_lda(
    corpus,
    stop_words,
    n_components=10
)

print("LDA експерименти завершено.")

Запуск LDA (k=5)...
Запуск LDA (k=8)...
Запуск LDA (k=10)...
LDA експерименти завершено.


#7. Top words per topic

In [25]:
from sentiment.src.topic_utils import get_top_words

def display_all_results(model, vectorizer, name, k):
    print(f"\nРЕЗУЛЬТАТИ ДЛЯ: {name} (k={k})")
    topics = get_top_words(model, vectorizer, n_top_words=10)
    for topic, words in topics.items():
        print(f"{topic}: {', '.join(words)}")


# Вивід для LSA
display_all_results(lsa_model_5, lsa_vec_5, "LSA", 5)
display_all_results(lsa_model_8, lsa_vec_8, "LSA", 8)
display_all_results(lsa_model_10, lsa_vec_10, "LSA", 10)

# Вивід для LDA
display_all_results(lda_model_5, lda_vec_5, "LDA", 5)
display_all_results(lda_model_8, lda_vec_8, "LDA", 8)
display_all_results(lda_model_10, lda_vec_10, "LDA", 10)


РЕЗУЛЬТАТИ ДЛЯ: LSA (k=5)
Topic 0: товар, магазин, замовлення, телефон, доставка, розетка, сайт, центр, новий, гроші
Topic 1: телефон, сервісний, сервісний центр, центр, ремонт, сервіс, місяць, повернення, гарантія, повернути
Topic 2: телефон, магазин, цитрус, ціна, задоволений, консультант, смартфон, плівка, скло, покупок
Topic 3: телефон, кур, 00, єра, кур єра, дзвонити, передзвонити, доставка, ніхто, замовлення
Topic 4: доставка, кур, єра, кур єра, центр, сервісний, сервісний центр, гарантійний, єр, 00

РЕЗУЛЬТАТИ ДЛЯ: LSA (k=8)
Topic 0: товар, магазин, замовлення, телефон, доставка, розетка, сайт, центр, новий, гроші
Topic 1: телефон, сервісний, сервісний центр, центр, ремонт, сервіс, місяць, повернення, гарантія, повернути
Topic 2: телефон, магазин, цитрус, ціна, задоволений, консультант, смартфон, плівка, покупок, скло
Topic 3: телефон, кур, кур єра, єра, 00, дзвонити, передзвонити, доставка, замовлення, ніхто
Topic 4: доставка, кур, єра, кур єра, центр, сервісний, сервісний цен

#8. Top documents per topic

In [28]:
from sentiment.src.topic_utils import get_top_documents

def display_top_docs(matrix, corpus, name, k, n_docs=2):
    print(f"\nТОП-ДОКУМЕНТИ ДЛЯ: {name} (k={k})")
    top_docs = get_top_documents(matrix, corpus, n_top_docs=n_docs)
    for topic_idx, docs in top_docs.items():
        print(f"\n{topic_idx}:")
        for i, doc in enumerate(docs):
            print(f"  Doc {i+1}: {str(doc)[:250]}...")

# Вивід для всіх LSA експериментів
display_top_docs(lsa_matrix_5, df_filtered['lemma_text'], "LSA", 5)
display_top_docs(lsa_matrix_8, df_filtered['lemma_text'], "LSA", 8)
display_top_docs(lsa_matrix_10, df_filtered['lemma_text'], "LSA", 10)

# Вивід для всіх LDA експериментів
display_top_docs(lda_matrix_5, df_filtered['lemma_text'], "LDA", 5)
display_top_docs(lda_matrix_8, df_filtered['lemma_text'], "LDA", 8)
display_top_docs(lda_matrix_10, df_filtered['lemma_text'], "LDA", 10)


ТОП-ДОКУМЕНТИ ДЛЯ: LSA (k=5)

Topic 0:
  Doc 1: придбати 20 . 08 . 2015р . в інтернет - магазин « цитрус » ірhоnе 5с 16 gb whіtе аррlе rеf slіm bох іw ( новый ) . при отримання телефон у відділень новий пошта зіткнутися з наступна проблема - телефон не включатися . розраховувати на те , що він роз...
  Doc 2: добрий времен суток , співробітник даний організація ! у ми воно , на жаль , не добре !! почати по порядок . на ваш сайт бути оформлений заявка < id > телефон хіаоmі мі мах 3 / 32 gоld , вартість 5984 , 00 гр-н , 27 . 02 . 2017 рік забрати ми цей тел...

Topic 1:
  Doc 1: дуже поганий робота сервісний центр магазин . телефон ремонтувати майже 2 місяць . залишитися багато негатив від цей сервісний центр . в телефон бути поміняний ім’я і назад він ніхто не поміняти після ремонт . при звернення в сервісний центр завжди г...
  Doc 2: купити телефон в магазин цитрус , все працювати добре , але з час телефон почати частково не ловити зв’язок , а потім і постійно , до я не могти додзвон

#9. Manual interpretation

Для ручної інтерпретації обрано модель LDA (k=10), оскільки вона показала найкраще розділення текстів без явного дублювання, властивого LSA.

* **Topic 0: Захист прав споживачів**
  * *Пояснення:* Тема сформована навколо юридичної лексики. Назва обрана через слова «право», «споживач», «захист», «закон». Документи (Doc 1, Doc 2) підтверджують цей контекст, оскільки користувачі описують гарантійні випадки, звернення до сервісного центру та складання актів про неремонтопридатність, посилаючись на свої права.

* **Topic 1: Змішана тема (загальні покупки)**
  * *Пояснення:* Ця тема є занадто загальною і змішує 2 різні сюжети. Топ-слова містять загальну лексику («купувати», «ціна», «інтернет») та різні товари («диск», «планшет»), тоді як у підтверджуючих документах йдеться переважно про варильні поверхні.

* **Topic 2: Недокомплектація товару**
  * *Пояснення:* Назва відображає проблему відсутності деталей при покупці. Слова «кріплення», «неповний» ідеально співпадають із документами (Doc 1, Doc 2), де клієнти прямо скаржаться на відсутні кріплення для столу та деталі для інсталяції унітазу.

* **Topic 3: Оформлення замовлення і робота менеджерів**
  * *Пояснення:* Ця тема описує етап взаємодії клієнта з магазином до отримання товару. Це чітко видно зі слів «замовлення», «доставка», «менеджер», «передзвонити». Документи підтверджують це скаргами на непередзвон менеджерів та перенесення часу доставки.

* **Topic 4: Шумна тема**
  * *Пояснення:* Тема є шумною і не має єдиного змісту. Вона групує випадкові товари («навігатор», «гриль») із роками («2014»). Документи не мають спільного сюжету, окрім згадок конкретних дат оформлення замовлення, що робить тему неінтерпретованою.

* **Topic 5: Змішана тема (комплектуючі та різна техніка)**
  * *Пояснення:* Хоча слова чітко вказують на комп'ютерне залізо («відеокарт», «процесор», «пам'ять»), тема змішує різні сюжети в документах. У текстах фігурують смартфони Lenovo та блендери Bosch, що свідчить про розмитість фокусу.

* **Topic 6: Гарантійний ремонт комп'ютерної техніки**
  * *Пояснення:* Назва обрана через явний фокус на поломках електроніки. Слова «ноутбук», «ремонт», «сервісний», «гарантія» ідеально підкріплені документами про здачу материнських плат та ноутбуків до сервісного центру через технічний брак (згорілі мікросхеми, неробочі HDD).

* **Topic 7: Проблеми та затримки з поверненням коштів**
  * *Пояснення:* Тема відображає фінансовий негатив клієнтів при поверненні товару. Ключові слова «повернення», «гроші», «повернути» прямо підкріплюються документами, де клієнти з обуренням описують довге очікування повернення коштів (наприклад, 13997 грн) після відправки бракованого товару.

* **Topic 8: Стан товару при отриманні**
  * *Пояснення:* Тема фокусується на стані упаковок та підозрах клієнтів щодо вживаності товару. Слова «телефон», «коробка», «плівка» збігаються зі скаргами в документах на брудні коробки, зняті захисні плівки та видачу вживаних (б/у) ноутбуків під виглядом нових.

* **Topic 9: Змішана тема (невідповідність очікуванням)**
  * *Пояснення:* Ця тема змішує декілька сюжетів і є занадто загальною. Слова вказують на дитячі подарунки («навушник», «дитина», «машинка», «подарунок»), але документи описують інші проблеми: не ту вилку на перехіднику для iPhone та неправильний розмір взуття Crocs.

#10. “Bad topics” analysis

У цьому розділі проаналізовано "погані" теми, виявлені під час експериментів.

**1. Проблемна тема: LSA (k=5), Topic 3 та Topic 4**
* **Яка це тема:** Це `duplicate topic`. Теми майже повністю дублюють одна одну. Топ-документи (Doc 1 та Doc 2) для обох тем виявилися абсолютно ідентичними (відгуки про замовлення подарунка та проблеми з відсутністю дзвінка від менеджера).
* **Чому тема вийшла поганою:** Проблема у виборі кількості тем ($k$) для конкретно цієї моделі. Алгоритм LSA (на основі SVD) не зміг знайти достатньо унікальних векторів для 5 різних тем у цьому корпусі, тому він просто "роздвоїв" один сильний сюжет про доставку на дві теми.
* **Що б ви змінили далі:**
  * Змінити $k$: зменшити кількість тем для моделі LSA, щоб уникнути дублювання.
  * Перейти на модель LDA, яка в сусідніх експериментах показала кращу здатність розділяти сюжети без дублікатів.

**2. Проблемна тема: LDA (k=10), Topic 4**
* **Яка це тема:** Це `mixed topic` та `domain noise topic`. Топ-слова містять цифри ("2014") та абсолютно не пов'язані товари ("навігатор", "гриль"). Топ-документи підтверджують змішування: один відгук стосується покупки навігатора, а інший — гриля. У них немає спільного логічного сюжету, окрім згадок конкретних дат замовлення.
* **Чому тема вийшла поганою:**
  * Проблема в `preprocessing`: у текстах залишилися специфічні дати і цифри, які алгоритм сприйняв як важливі маркери теми.
  * Проблема в неоднорідності корпусу: датасет містить відгуки на дуже різні категорії товарів (від дрібної побутової техніки до електроніки). При великому $k$ модель починає створювати штучні "збірні" теми з рідкісних товарів.
* **Що б ви змінили далі:**
  * Покращити `preprocessing`: перед векторизацією прибрати цифри, дати та технічні артикули за допомогою регулярних виразів.
  * Обмежити корпус однією підтемою: наприклад, залишити тільки відгуки про смартфони та ноутбуки. Це змусить алгоритм шукати приховані теми саме у проблемах клієнтів (ремонт, доставка), а не просто групувати відгуки за різними типами товарів.
  * Змінити `min_df`: збільшити поріг, щоб відкинути дуже специфічні назви рідкісних товарів, які створюють шум.

#11. LSA vs LDA comparison

**1. Читабельність тем**

Модель **LDA** згенерувала значно більш читабельні та логічно завершені теми. Її результати легко інтерпретувати як конкретні процеси (наприклад, гарантійний ремонт чи повернення коштів). Теми LSA натомість виявилися більш загальними та розмитими.

**2. Аналіз недоліків (дублікати, шум, змішані теми)**
* **Дубльовані теми:** Очевидна проблема моделі **LSA**. При $k=5$ вона створила дві абсолютно ідентичні теми (Topic 3 та 4), що підтверджено однаковими топ-документами. У LDA дублікатів практично немає.
* **Змішані теми:** Значно частіше зустрічаються в **LSA**, оскільки вона групує тексти навколо загальновживаних слів («телефон», «магазин»). LDA теж має змішані сюжети (Topic 5 при $k=10$), але їх відсоток менший.
* **Шумні теми:** **LDA** виявилася більш чутливою до специфічного технічного шуму (наприклад, виділила окрему тему для дат та цифр), тоді як LSA такі нюанси згладжує.

**3. Яка модель корисніша для нашого корпусу**

Для даного корпусу клієнтських відгуків магазинів електроніки модель LDA виявилася набагато кориснішою та ефективнішою. Завдяки ймовірнісному підходу вона змогла чітко розпізнати вузькі клієнтські болі: скарги на недокомплектацію (відсутні кріплення), складнощі з гарантійним ремонтом комп'ютерної техніки та тривале очікування повернення коштів. LSA у цьому ж корпусі продемонструвала схильність до дублювання та створення надто загальних сюжетів, що не дають реального розуміння бізнес-проблем. LDA дозволила зафіксувати навіть використання специфічної юридичної лексики (захист прав споживачів), що є цінним інсайтом. Тому саме LDA є надійнішим інструментом для практичної аналітики цих відгуків.

**Оцінка Coherence**

Для математичної перевірки якості моделей було розраховано метрику Coherence. Модель LDA показала такі результати: k=5 (0.4514), k=8 (**0.5128**), k=10 (0.4824). Показники LSA виявилися нижчими: k=5 (0.4599), k=8 (**0.4721**), k=10 (0.4561).

Перевага LDA у максимальному показнику узгодженості (0.5128 проти 0.4721 у LSA) математично підтверджує нашу ручну оцінку: слова в темах LDA дійсно краще пов'язані між собою в реальних документах. Крім того, хоча математичний пік для LDA досягається при k=8, ручна інтерпретація показала краще змістове розділення саме при k=10. Це доводить, що метрика Coherence є чудовим додатковим сигналом якості моделі, але фінальний висновок завжди має базуватися на ручній оцінці топ-слів та топ-документів.

In [32]:
!pip install gensim -q
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sentiment.src.topic_utils import get_top_words

texts = [str(text).split() for text in df_filtered['lemma_text']]
dictionary = Dictionary(texts)

# Функція для розрахунку Coherence
def calculate_coherence_score(model, vectorizer, name, k):
    topics_dict = get_top_words(model, vectorizer, n_top_words=10)
    topics_list = [words for _, words in topics_dict.items()]
    cm = CoherenceModel(topics=topics_list, texts=texts, dictionary=dictionary, coherence='c_v')
    score = cm.get_coherence()
    print(f"Coherence Score для {name} (k={k}): {score:.4f}")
    return score

print("Оцінка Coherence для LDA")
calculate_coherence_score(lda_model_5, lda_vec_5, "LDA", 5)
calculate_coherence_score(lda_model_8, lda_vec_8, "LDA", 8)
calculate_coherence_score(lda_model_10, lda_vec_10, "LDA", 10)

print("\nОцінка Coherence для LSA")
calculate_coherence_score(lsa_model_5, lsa_vec_5, "LSA", 5)
calculate_coherence_score(lsa_model_8, lsa_vec_8, "LSA", 8)
calculate_coherence_score(lsa_model_10, lsa_vec_10, "LSA", 10)

Оцінка Coherence для LDA
Coherence Score для LDA (k=5): 0.4514
Coherence Score для LDA (k=8): 0.5128
Coherence Score для LDA (k=10): 0.4824

Оцінка Coherence для LSA
Coherence Score для LSA (k=5): 0.4599
Coherence Score для LSA (k=8): 0.4721
Coherence Score для LSA (k=10): 0.4561


np.float64(0.4561186795086371)

#12. Generate docs/audit_summary_lab8.md

In [33]:
import os

os.makedirs('docs', exist_ok=True)

audit_summary_text = """# Audit Summary Lab 8: Topic Modeling

**1. Розмір корпусу після фільтрації:**
3034 документи (фільтрація не видалила жодного документа, оскільки датасет вже був попередньо очищений від порожніх та занадто коротких значень).

**2. Які моделі протестовано:**
* LSA (Latent Semantic Analysis)
* LDA (Latent Dirichlet Allocation)

**3. Які k (кількість тем) протестовано:**
Для обох моделей перевірено k = 5, 8 та 10.

**4. 2–3 найкращі теми (на основі LDA, k=10):**
* **"Недокомплектація товару"**: чітко виділена проблема відсутності кріплень та деталей при доставці.
* **"Гарантійний ремонт"**: сфокусована тема про поломки комп'ютерної техніки (згорілі мікросхеми, неробочі диски) та взаємодію з сервісними центрами.
* **"Затримки з поверненням коштів"**: зібрала скарги клієнтів на тривале очікування фінансів після повернення бракованого товару.

**5. 2–3 найгірші теми:**
* **LSA (k=5), Теми 3 та 4**: Duplicate topics. Абсолютно ідентичні теми, підкріплені одними й тими самими відгуками.
* **LDA (k=10), Тема 4**: Domain noise / Mixed topic. Тема згрупувала випадкові дати ("2014") та абсолютно різні товари (навігатори, грилі).

**6. Що їх зіпсувало:**
* **LSA (дублікати)**: зіпсувалася через те, що алгоритму (на основі SVD) не вистачило унікальних векторів для розбиття на 5 тем, тому він просто "роздвоїв" один сильний сюжет.
* **LDA (шумна тема)**: зіпсувалася через недостатній preprocessing (у текстах залишилися специфічні дати і цифри) та загальну неоднорідність корпусу (дуже багато різних категорій товарів).

**7. Яка модель краща саме для вашого кейсу:**
Для аналізу відгуків магазинів електроніки значно кращою виявилася **LDA**. LSA була схильна до створення дублікатів та надто загальних сюжетів, з яких важко дістати бізнес-інсайти. Натомість LDA змогла знайти вузькі, конкретні проблеми клієнтів (комплектація, ремонт, повернення грошей). Це підтвердилося і метрикою Coherence: найкращий показник LDA (0.5128 при k=8) перевищив найкращий показник LSA (0.4721 при k=8).

**8. Що б ви робили далі:**
* **Покращення Preprocessing**: використала б регулярні вирази, щоб видалити всі дати, цифри та технічні артикули перед векторизацією.
* **Обмеження корпусу**: сегментувала б датасет за підтемами (наприклад, окремо моделювала б відгуки тільки про смартфони, окремо — про побутову техніку), щоб уникнути змішування різних сутностей.
* **Тюнінг параметрів**: збільшила б поріг `min_df`, щоб відкинути назви рідкісних товарів, які створюють шум у темах.
"""

# Записуємо текст у файл
with open('docs/audit_summary_lab8.md', 'w', encoding='utf-8') as f:
    f.write(audit_summary_text)

print("Файл docs/audit_summary_lab8.md згенеровано.")

Файл docs/audit_summary_lab8.md згенеровано.
